<p align="center">
  <img src="https://raw.githubusercontent.com/VK-Ant/sightrag/main/assets/sightrag_banner.png" alt="SightRAG Banner" width="100%">
</p>

# SightRAG v0.2
### See. Search. Retrieve.

<p align="center">
  <a href="https://pypi.org/project/sightrag/"><img src="https://img.shields.io/pypi/v/sightrag" alt="PyPI"></a>
  <a href="https://github.com/VK-Ant/sightrag/blob/main/LICENSE"><img src="https://img.shields.io/badge/license-Apache%202.0-blue" alt="License"></a>
  <a href="https://www.python.org/"><img src="https://img.shields.io/badge/python-3.9+-green" alt="Python"></a>
</p>

**What's new in v0.2:**
- Auto backend selection (ONNX/TensorRT/OpenVINO/PyTorch)
- Pluggable models (bring your own detector/embedder)
- `rag.show()` — visualize results with bounding boxes
- C++ core for faster data pipeline

**GitHub:** [VK-Ant/sightrag](https://github.com/VK-Ant/sightrag) | **PyPI:** [pip install sightrag](https://pypi.org/project/sightrag/) | **Author:** Ant (VK-Ant)

## Step 1 — Install

In [ ]:
!pip install sightrag -q

from sightrag import SightRAG, __version__
print(f'\nSightRAG v{__version__} installed')

## Step 2 — Create Demo Data

In [ ]:
from PIL import Image, ImageDraw
import os, matplotlib.pyplot as plt

for d in ['demo/input', 'demo/reference']:
    os.makedirs(d, exist_ok=True)

configs = [
    ('shelf_empty', (240,240,240), [(50,80,100,150,(255,0,0)), (120,80,170,150,(0,0,255))]),
    ('shelf_full', (240,240,240), [(50+i*70,80,100+i*70,150,(i*30,100,200)) for i in range(8)]),
    ('person_door', (200,200,220), [(250,100,310,160,(255,200,150)), (265,160,295,300,(0,0,180))]),
    ('person_stand', (180,220,180), [(280,80,360,160,(255,200,150)), (295,160,345,340,(200,0,0))]),
    ('parking', (80,80,80), [(50,280,120,430,(255,0,0)), (160,280,230,430,(0,0,255))]),
    ('room', (230,230,230), [(200,50,440,250,(135,206,250)), (450,250,550,350,(139,69,19))]),
]

for name, bg, rects in configs:
    img = Image.new('RGB', (640, 480), bg)
    draw = ImageDraw.Draw(img)
    if 'shelf' in name:
        draw.rectangle([20, 150, 620, 160], fill=(139, 69, 19))
    for r in rects:
        draw.rectangle(list(r[:4]), fill=r[4])
    img.save(f'demo/input/{name}.jpg')

# Reference images
img = Image.new('RGB', (200, 300), (180,220,180))
draw = ImageDraw.Draw(img)
draw.ellipse([70,20,130,80], fill=(255,200,150))
draw.rectangle([80,80,120,200], fill=(200,0,0))
img.save('demo/reference/ref_person.jpg')

img = Image.new('RGB', (320, 240), (240,240,240))
draw = ImageDraw.Draw(img)
draw.rectangle([10,80,310,88], fill=(139,69,19))
draw.rectangle([30,30,80,80], fill=(255,0,0))
img.save('demo/reference/ref_shelf.jpg')

print(f'Input: {len(os.listdir("demo/input"))} images')
print(f'Reference: {len(os.listdir("demo/reference"))} images')

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, f in zip(axes, sorted(os.listdir('demo/input'))):
    ax.imshow(Image.open(f'demo/input/{f}'))
    ax.set_title(f.replace('.jpg',''), fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3 — Index + Query (3 lines)

In [ ]:
from sightrag import SightRAG

rag = SightRAG()
rag.index('./demo/input/')
results = rag.query('find empty shelf')

for i, r in enumerate(results[:3], 1):
    print(f"{i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f} | {r['label']}")

## Step 4 — Text Queries

In [ ]:
for q in ['find person', 'find car', 'find room with furniture', 'find shelf']:
    results = rag.query(q, top_k=2)
    print(f'\n"{q}"')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")

## Step 5 — Reference Image Query

In [ ]:
for ref in sorted(os.listdir('demo/reference')):
    results = rag.query(reference=f'demo/reference/{ref}', top_k=3)
    print(f'\nReference: {ref}')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")

## Step 6 — Visualize Results (NEW in v0.2)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def visualize(query, results, ref_img=None, max_show=3):
    n = min(len(results), max_show)
    cols = n + (1 if ref_img else 0)
    fig, axes = plt.subplots(1, cols, figsize=(5*cols, 5))
    if cols == 1: axes = [axes]
    start = 0
    if ref_img:
        axes[0].imshow(Image.open(ref_img))
        axes[0].set_title('REFERENCE', color='red', fontweight='bold')
        axes[0].axis('off')
        start = 1
    for idx, r in enumerate(results[:n]):
        img = Image.open(r['image_path']).copy()
        draw = ImageDraw.Draw(img)
        bbox = r['bbox']
        if bbox and len(bbox) == 4:
            draw.rectangle(bbox, outline='red', width=3)
            draw.text((bbox[0]+2, bbox[1]-12), f"{r['label']} ({r['score']:.2f})", fill='red')
        axes[start+idx].imshow(img)
        axes[start+idx].set_title(f"{r['image_path'].split('/')[-1]}\nscore: {r['score']:.4f}", fontsize=10)
        axes[start+idx].axis('off')
    plt.suptitle(f'Query: "{query}"' if not ref_img else f'Reference query', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize('find person', rag.query('find person', top_k=3))
visualize('find shelf', rag.query('find shelf', top_k=3))
visualize('', rag.query(reference='demo/reference/ref_person.jpg', top_k=3), ref_img='demo/reference/ref_person.jpg')

## Step 7 — rag.show() saves annotated images

In [ ]:
os.makedirs('output', exist_ok=True)
results = rag.query('find person', top_k=3)
rag.show(results, save='./output/')

# Display saved annotated images
saved = sorted(os.listdir('output'))
if saved:
    fig, axes = plt.subplots(1, len(saved), figsize=(5*len(saved), 5))
    if len(saved) == 1: axes = [axes]
    for ax, f in zip(axes, saved):
        ax.imshow(Image.open(f'output/{f}'))
        ax.set_title(f, fontsize=9)
        ax.axis('off')
    plt.suptitle('rag.show() output', fontsize=14)
    plt.tight_layout()
    plt.show()

## Step 8 — Upload Your Own Images

In [ ]:
try:
    from google.colab import files
    print('Upload your images:')
    uploaded = files.upload()
    os.makedirs('my_images', exist_ok=True)
    for name, data in uploaded.items():
        with open(f'my_images/{name}', 'wb') as f: f.write(data)
    my_rag = SightRAG(index_path='./my_index')
    my_rag.index('./my_images/')
    query = input('Enter search query: ')
    results = my_rag.query(query, top_k=3)
    visualize(query, results)
    my_rag.clear()
except ImportError:
    print('File upload only in Google Colab')

## Step 9 — Check Backend

In [ ]:
print(f'SightRAG: {rag}')
print(f'Backend: {rag._backend.name}')
print(f'Embed dim: {rag._backend.embed_dim}')

# Install ONNX for 2x speed boost
# !pip install onnxruntime -q
# Then re-initialize: rag = SightRAG()  # auto-selects ONNX

## Cleanup

In [ ]:
import shutil
rag.clear()
shutil.rmtree('./output', ignore_errors=True)

print('=' * 50)
print('  SightRAG v0.2 Demo Complete!')
print('=' * 50)
print()
print('  Tested:')
print('  - Image folder indexing')
print('  - Text queries')
print('  - Reference image queries')
print('  - rag.show() visualization')
print('  - Upload your own images')
print()
print('  GitHub: github.com/VK-Ant/sightrag')
print('  PyPI: pip install sightrag')
print('  Built by Ant (VK-Ant)')